# Set up and read data

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("EDA Test") \
    .getOrCreate()

df = spark.read.json(
    "hdfs://localhost:9000/ev-project/data/bronze/ev_sessions/caltech/year=2021/month=09/day=13"
)

In [29]:
df.show(5)

+---------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+------------+--------------------+------+--------+------------+-------------------+---------+--------------------+
|      _batch_id|                 _id|        _ingest_time|clusterID|      connectionTime|      disconnectTime|    doneChargingTime|kWhDelivered|           sessionID|siteID| spaceID|   stationID|           timezone|   userID|          userInputs|
+---------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+------------+--------------------+------+--------+------------+-------------------+---------+--------------------+
|20260407_141136|61550519f9af8b769...|2026-04-07T14:11:...|     0039|2021-09-13 18:52:...|2021-09-13 20:05:...|                NULL|      45.064|2_39_81_4550_2021...|  0002|11900388|2-39-81-4550|America/Los_Angeles|000019055|[{286, 28.6, 100,...|
|20260407_14

In [30]:
df.printSchema()

root
 |-- _batch_id: string (nullable = true)
 |-- _id: string (nullable = true)
 |-- _ingest_time: string (nullable = true)
 |-- clusterID: string (nullable = true)
 |-- connectionTime: string (nullable = true)
 |-- disconnectTime: string (nullable = true)
 |-- doneChargingTime: string (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- sessionID: string (nullable = true)
 |-- siteID: string (nullable = true)
 |-- spaceID: string (nullable = true)
 |-- stationID: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userID: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double (nullable = true)
 |    |    |-- milesRequested: long (nullable = true)
 |    |    |-- minutesAvailable: long (nullable = true)
 |    |    |-- modifiedAt: string (nullable = true)
 |    |    |-- paymentRequired: boolean (nullable = true)
 

In [31]:
df.count()

29

In [32]:
df.select("stationID").distinct().count()

19

remove ID columns

In [4]:
cols_to_drop = [
    "_id", "sessionID", "stationID", "spaceID",
    "siteID", "clusterID", "userID"
]

df = df.drop(*cols_to_drop)

In [9]:
df.printSchema()

root
 |-- _batch_id: string (nullable = true)
 |-- _id: string (nullable = true)
 |-- _ingest_time: string (nullable = true)
 |-- clusterID: string (nullable = true)
 |-- connectionTime: string (nullable = true)
 |-- disconnectTime: string (nullable = true)
 |-- doneChargingTime: string (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- sessionID: string (nullable = true)
 |-- siteID: string (nullable = true)
 |-- spaceID: string (nullable = true)
 |-- stationID: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userID: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double (nullable = true)
 |    |    |-- milesRequested: long (nullable = true)
 |    |    |-- minutesAvailable: long (nullable = true)
 |    |    |-- modifiedAt: string (nullable = true)
 |    |    |-- paymentRequired: boolean (nullable = true)
 

convert to timestamp (timezone-naive)

df = df.withColumn("connectionTime", to_timestamp("connectionTime")) \
       .withColumn("disconnectTime", to_timestamp("disconnectTime")) \
       .withColumn("doneChargingTime", to_timestamp("doneChargingTime"))

Time features 

In [10]:
df = df.withColumn("hour", hour("connectionTime")) \
       .withColumn("day_of_week", dayofweek("connectionTime") - 1) \
       .withColumn("month", month("connectionTime"))

Weekend + Season features

In [11]:
df = df.withColumn(
    "is_weekend",
    when(col("day_of_week").isin([5, 6]), 1).otherwise(0)
)

df = df.withColumn(
    "season",
    when(col("month").isin([12,1,2]), 0)  # Winter
    .when(col("month").isin([3,4,5]), 1)  # Spring
    .when(col("month").isin([6,7,8]), 2)  # Summer
    .otherwise(3)                         # Fall
)

Cyclical features

In [12]:
df = df.withColumn("hour_sin", sin(2 * 3.14159265359 * col("hour") / 24)) \
       .withColumn("hour_cos", cos(2 * 3.14159265359 * col("hour") / 24))

Calendar features

In [13]:
df = df.withColumn("day_of_year", dayofyear("connectionTime")) \
       .withColumn("week_of_year", weekofyear("connectionTime"))

Holiday feature

In [14]:
df = df.withColumn(
    "is_holiday",
    when(col("month").isin([12, 1]), 1).otherwise(0)
)

Duration features

In [15]:
df = df.withColumn(
    "duration",
    (unix_timestamp("disconnectTime") - unix_timestamp("connectionTime")) / 3600
)

df = df.withColumn(
    "charging_duration",
    (unix_timestamp("doneChargingTime") - unix_timestamp("connectionTime")) / 3600
)

In [ ]:
Log transform

In [16]:
df = df.withColumn(
    "charging_duration_log",
    log1p(col("charging_duration"))
)

5. Interaction Features

In [17]:
df = df.withColumn(
    "hour_charging_interaction",
    col("hour") * col("charging_duration")
)

df = df.withColumn(
    "weekend_charging_interaction",
    col("is_weekend") * col("charging_duration")
)

Lag + Rolling

In [22]:
# bạn PHẢI define window theo time
window_spec = Window.orderBy("connectionTime")

# Lag features
df = df.withColumn("lag_1_log", lag("charging_duration_log", 1).over(window_spec)) \
       .withColumn("lag_2_log", lag("charging_duration_log", 2).over(window_spec)) \
       .withColumn("lag_3_log", lag("charging_duration_log", 3).over(window_spec))


Outlier handling (IQR cap)

In [23]:
quantiles = df.approxQuantile(
    ["duration", "charging_duration", "kWhDelivered"],
    [0.25, 0.75],
    0.01
)

cols = ["duration", "charging_duration", "kWhDelivered"]

for i, c in enumerate(cols):
    q1 = quantiles[i][0]
    q3 = quantiles[i][1]
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    df = df.withColumn(
        c,
        when(col(c) < lower, lower)
        .when(col(c) > upper, upper)
        .otherwise(col(c))
    )

26/04/07 14:51:46 ERROR Executor: Exception in task 0.0 in stage 4.0 (TID 4)/ 1]
org.apache.spark.SparkUpgradeException: [INCONSISTENT_BEHAVIOR_CROSS_VERSION.PARSE_DATETIME_BY_NEW_PARSER] You may get a different result due to the upgrading to Spark >= 3.0:
Fail to parse '2021-09-13 20:05:10-07:00' in the new parser. You can set "spark.sql.legacy.timeParserPolicy" to "LEGACY" to restore the behavior before Spark 3.0, or set to "CORRECTED" and treat it as an invalid datetime string.
	at org.apache.spark.sql.errors.ExecutionErrors.failToParseDateTimeInNewParserError(ExecutionErrors.scala:54)
	at org.apache.spark.sql.errors.ExecutionErrors.failToParseDateTimeInNewParserError$(ExecutionErrors.scala:48)
	at org.apache.spark.sql.errors.ExecutionErrors$.failToParseDateTimeInNewParserError(ExecutionErrors.scala:218)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:142)
	at org.apache.spark.sql.catalyst.util.DateT

Py4JJavaError: An error occurred while calling o191.approxQuantile.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 4.0 failed 1 times, most recent failure: Lost task 0.0 in stage 4.0 (TID 4) (192.168.40.130 executor driver): org.apache.spark.SparkUpgradeException: [INCONSISTENT_BEHAVIOR_CROSS_VERSION.PARSE_DATETIME_BY_NEW_PARSER] You may get a different result due to the upgrading to Spark >= 3.0:
Fail to parse '2021-09-13 20:05:10-07:00' in the new parser. You can set "spark.sql.legacy.timeParserPolicy" to "LEGACY" to restore the behavior before Spark 3.0, or set to "CORRECTED" and treat it as an invalid datetime string.
	at org.apache.spark.sql.errors.ExecutionErrors.failToParseDateTimeInNewParserError(ExecutionErrors.scala:54)
	at org.apache.spark.sql.errors.ExecutionErrors.failToParseDateTimeInNewParserError$(ExecutionErrors.scala:48)
	at org.apache.spark.sql.errors.ExecutionErrors$.failToParseDateTimeInNewParserError(ExecutionErrors.scala:218)

Handle missing values (AFTER lag)

In [24]:
df = df.dropna()

9. Target transform

In [25]:
df = df.withColumn(
    "kWhDelivered_log",
    log1p(col("kWhDelivered"))
)

Final feature selection (Silver dataset)

In [26]:
final_cols = [
    "hour", "day_of_week", "month", "season",
    "duration", "charging_duration", "charging_duration_log",
    "hour_sin", "hour_cos", "day_of_year", "week_of_year",
    "is_holiday",
    "lag_1_log", "lag_2_log", "lag_3_log",
    "rolling_mean_3_log", "rolling_mean_5_log",
    "kWhDelivered_log"
]

df_final = df.select(*final_cols)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `rolling_mean_3_log` cannot be resolved. Did you mean one of the following? [`lag_3_log`, `lag_1_log`, `lag_2_log`, `_ingest_time`, `charging_duration_log`].;
'Project [hour#238, day_of_week#255, month#273, season#312, duration#453, charging_duration#480, charging_duration_log#508, hour_sin#333, hour_cos#355, day_of_year#378, week_of_year#402, is_holiday#427, lag_1_log#660, lag_2_log#693, lag_3_log#727, 'rolling_mean_3_log, 'rolling_mean_5_log, kWhDelivered_log#855]
+- Project [_batch_id#131, _id#132, _ingest_time#133, clusterID#134, connectionTime#135, disconnectTime#136, doneChargingTime#137, kWhDelivered#138, sessionID#139, siteID#140, spaceID#141, stationID#142, timezone#143, userID#144, userInputs#145, hour#238, day_of_week#255, month#273, is_weekend#292, season#312, hour_sin#333, hour_cos#355, day_of_year#378, week_of_year#402, ... 10 more fields]
   +- Filter atleastnnonnulls(34, _bat

In [ ]:
# Write to HDFS (Silver layer)

In [20]:
df_final.write.mode("overwrite").parquet(
    "hdfs://localhost:9000/ev-project/data/silver/ev-sessions/caltech"
)

NameError: name 'df_final' is not defined